# Step 8 · Snapshot, conversion, and row-accounting audit

This notebook separates source snapshot verification from deterministic conversion and validation. The source snapshot is immutable. Converted rows remain in a distinct V4 staging tree until every source row is accounted for and the report has been reviewed.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, log_step
ROOT, LOG, LOG_PATH = configure_notebook('08_audit_migration', 'migration-audit')
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-02 11:29:50,702 INFO Notebook session started at 2026-11-02T11:29:50.702623+00:00


2026-11-02 11:29:50,702 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 11:29:50,703 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/migration-audit/05_migration_audit.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/migration-audit/05_migration_audit.log


## Step 1 · Review snapshot identity and checksum manifest

In [2]:
import json
from IPython.display import display
SNAPSHOT_ROOT = ROOT / 'data/migration/v3_snapshot'
MANIFEST_PATH = SNAPSHOT_ROOT / 'transfer_manifest.json'
manifest = json.loads(MANIFEST_PATH.read_text())
print('Snapshot:', SNAPSHOT_ROOT)
display({key: manifest.get(key) for key in ('source_commit','file_count','total_bytes','created_utc')})
print('Snapshot data exists:', (SNAPSHOT_ROOT / 'data').exists())
LOG.info('Snapshot contains %s files and %s bytes', manifest['file_count'], manifest['total_bytes'])

Snapshot: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/migration/v3_snapshot


{'source_commit': '59409a9136bb1ce8477bc80179c853a9132188a8',
 'file_count': 147,
 'total_bytes': 4585858645,
 'created_utc': '2026-11-02T10:27:25.400821+00:00'}

2026-11-02 11:29:50,707 INFO Snapshot contains 147 files and 4585858645 bytes


Snapshot data exists: True


## Step 2 · Verify every snapshot checksum

In [3]:
from pathogen_atlas_v4.migration.import_v3_data import verify_manifest
VERIFY_SNAPSHOT = False  # set True to re-hash the full 4.6 GB snapshot
if VERIFY_SNAPSHOT:
    verification = verify_manifest(manifest, SNAPSHOT_ROOT / 'data')
    display(verification)
    LOG.info('Snapshot verification result: %s', verification)
    if not verification.get('ok'):
        raise RuntimeError('Snapshot verification failed; do not convert or promote this snapshot.')
else:
    print('Checksum pass skipped. Set VERIFY_SNAPSHOT=True to hash every source file.')
    LOG.info('Full snapshot checksum verification skipped by notebook setting')

2026-11-02 11:29:51,134 INFO Full snapshot checksum verification skipped by notebook setting


Checksum pass skipped. Set VERIFY_SNAPSHOT=True to hash every source file.


## Step 3 · Inspect deterministic conversion reports

In [4]:
import pandas as pd
REPORTS = sorted((ROOT / 'data/v4/staging').glob('*/migration_report.json'))
reports = [(path, json.loads(path.read_text())) for path in REPORTS]
print('Migration reports found:', len(reports))
if reports:
    report_path, report = reports[-1]
    print('Selected staging:', report_path.parent.name)
    print('Source commit:', report.get('source_commit'), '| config:', report.get('migration_config_version'))
    table_rows = [{'table': name, **counts} for name, counts in report.get('tables', {}).items()]
    display(pd.DataFrame(table_rows).set_index('table'))
    print('Warnings:', len(report.get('warnings', [])), '| rejected detail rows:', len(report.get('rejected_rows', [])))
    print('Metadata conflicts:', len(report.get('article_metadata_conflicts', [])))
    print('Duplicate counts:', report.get('duplicate_counts', {}))
    print('Unmapped counts:', report.get('unmapped_counts', {}))
    LOG.info('Selected migration report %s; tables=%d warnings=%d', report_path, len(table_rows), len(report.get('warnings', [])))
else:
    report = {}
    print('No migration report found.')


Migration reports found: 3
Selected staging: v3-to-v4-20261102-ncbi-f278697a8ba6-v3
Source commit: 59409a9136bb1ce8477bc80179c853a9132188a8 | config: v3-to-v4-import-v3-ncbi-f278697a8ba6


,source_rows,migrated_rows,unmapped_rows,duplicate_rows,conflicting_pmids,source_abstract_rows,search_match_pmids_without_fetched_abstract,placeholder_article_rows,output_rows_including_placeholders,unmapped_pathogen_rows,...,rejected_rows,v3_ledger_rows,v2_legacy_rows,evidence_span_available,evidence_span_type,source_files,indexed_files,jsonl_records_indexed,jsonl_pmids_indexed,malformed_jsonl_records
table,,,,,,,,,,,,,,,,,,,,,
taxonomy/pathogens.parquet,142201.0,142201.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
taxonomy/pathogen_aliases.parquet,233550.0,233550.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
taxonomy/pathogen_variants.parquet,19285.0,19285.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
literature/articles.parquet,163687.0,1733750.0,NaN,0.0,0.0,163687.0,1570063.0,1570063.0,1733750.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
literature/pathogen_article_matches.parquet,3450866.0,3450866.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
taxonomy/hosts.parquet,5.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
decisions/legacy_v3_decisions.parquet,2344486.0,2344486.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,209752.0,2134734.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
decisions/legacy_v3_feature_assessments.parquet,510641.0,510641.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
evidence/pathogen_host_evidence.parquet,2674.0,2674.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,True,full_abstract_legacy,NaN,NaN,NaN,NaN,NaN


2026-11-02 11:29:51,149 INFO Selected migration report /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/staging/v3-to-v4-20261102-ncbi-f278697a8ba6-v3/migration_report.json; tables=10 warnings=0


Warnings: 0 | rejected detail rows: 0
Metadata conflicts: 0
Duplicate counts: {'articles': 0, 'pathogen_article_match_keys': 0}
Unmapped counts: {'pathogen_aliases': 0, 'literature_match_pathogen': 0, 'literature_match_article': 0, 'accepted_legacy_evidence': 0, 'accepted_legacy_evidence_missing_abstract': 0}


## Step 4 · Check staging tables and linked IDs

In [5]:
import pandas as pd
REPORT_PATHS = sorted((ROOT / 'data/v4/staging').glob('*/migration_report.json'))
STAGING_ROOT = REPORT_PATHS[-1].parent if REPORT_PATHS else None
if STAGING_ROOT:
    pathogens = pd.read_parquet(STAGING_ROOT / 'taxonomy/pathogens.parquet')
    hosts = pd.read_parquet(STAGING_ROOT / 'taxonomy/hosts.parquet')
    articles = pd.read_parquet(STAGING_ROOT / 'literature/articles.parquet')
    matches = pd.read_parquet(STAGING_ROOT / 'literature/pathogen_article_matches.parquet')
    evidence = pd.read_parquet(STAGING_ROOT / 'evidence/pathogen_host_evidence.parquet')
    checks = {'pathogens': len(pathogens), 'hosts': len(hosts), 'articles': len(articles),
              'matches': len(matches), 'legacy_supported_pairs': int(evidence.decision.eq('supported').sum()),
              'unique_pathogen_ids': pathogens.pathogen_id.is_unique,
              'unknown_match_pathogens': int((~matches.pathogen_id.isin(set(pathogens.pathogen_id))).sum()),
              'unknown_match_pmids': int((~matches.pmid.astype(str).isin(set(articles.pmid.astype(str)))).sum())}
    display(checks)
    if checks['unknown_match_pathogens'] or checks['unknown_match_pmids'] or not checks['unique_pathogen_ids']:
        raise ValueError('Staging reference checks failed')
    LOG.info('Staging references validated: %s', checks)
else:
    print('No staging conversion report found.')

{'pathogens': 142201,
 'hosts': 5,
 'articles': 1733750,
 'matches': 3450866,
 'legacy_supported_pairs': 2674,
 'unique_pathogen_ids': True,
 'unknown_match_pathogens': 0,
 'unknown_match_pmids': 0}

2026-11-02 11:29:53,849 INFO Staging references validated: {'pathogens': 142201, 'hosts': 5, 'articles': 1733750, 'matches': 3450866, 'legacy_supported_pairs': 2674, 'unique_pathogen_ids': True, 'unknown_match_pathogens': 0, 'unknown_match_pmids': 0}


## Step 5 · Review disposition before promotion

In [6]:
PROMOTE_STAGING = False
if PROMOTE_STAGING:
    raise RuntimeError('Promotion requires independent review of row accounting, foreign keys, legacy provenance, and the migration report.')
print('Staging remains staged. Review the report and all unresolved/rejected/duplicate categories before promotion.')
LOG.info('Promotion state: not promoted')

2026-11-02 11:29:53,853 INFO Promotion state: not promoted


Staging remains staged. Review the report and all unresolved/rejected/duplicate categories before promotion.
